# Projet de Détection de la Maladie d'Alzheimer par Signaux EEG
Ce notebook implémente un pipeline complet incluant le téléchargement du dataset OpenNeuro ds004504, l'extraction de caractéristiques non-linéaires et la classification par Machine Learning.

In [ ]:
import mne
import mne_bids
import os

# Résolution de la racine du dataset (uniquement data\ds004504 en local)
candidate_roots = [
    os.path.join('..', 'data', 'ds004504'),
    os.path.join('.', 'data', 'ds004504'),
]
data_root = None
for cand in candidate_roots:
    if os.path.exists(cand):
        data_root = cand
        break

if data_root is None:
    print("Erreur: impossible de localiser la racine des données. Vérifiez la présence de data\\ds004504 (relatif au notebook).")
else:
    print(f"Racine des données détectée: {os.path.abspath(data_root)}")

# Paramètres du sujet d'exemple (ds004504)
subject_id = '001'
task_id = 'eyesclosed'

# Essayer d'abord de lire les données prétraitées EEGLAB (.set) dans derivatives
raw = None
if data_root is not None:
    eeg_dir = os.path.join(data_root, 'derivatives', f'sub-{subject_id}', 'eeg')
    if os.path.isdir(eeg_dir):
        set_files = [f for f in os.listdir(eeg_dir) if f.endswith('_eeg.set')]
        if set_files:
            set_path = os.path.join(eeg_dir, set_files[0])
            print(f"Lecture du fichier EEGLAB: {set_path}")
            try:
                raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
            except Exception as e:
                print(f"Erreur de lecture EEGLAB: {e}")

# Si échec, tenter une lecture BIDS classique (si structure BIDS brute disponible)
if raw is None and data_root is not None:
    bids_path = mne_bids.BIDSPath(
        subject=subject_id,
        task=task_id,
        datatype='eeg',
        root=data_root
    )
    print(f"Tentative de lecture BIDS: {bids_path.fpath}")
    try:
        raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False)
    except Exception as e:
        print(f"Erreur de lecture BIDS: {e}")

# Normalisation de la fréquence d'échantillonnage
if raw is not None:
    try:
        raw.load_data()
        if raw.info['sfreq'] > 250:
            raw.resample(sfreq=250)
        print("\nRaw data loaded successfully:")
        print(raw)
    except Exception as e:
        print(f"Erreur lors du chargement ou du resampling: {e}")

## 1. Exemple d'extraction de Caractéristiques sur 1 Sujet
Traitement des données du dossier `derivatives\sub-001` :
- Chargement des fichiers EEGLAB (.set).
- Segmentation en epochs de 4 secondes.
- Calcul des caractéristiques : Higuchi Fractal Dimension, SVD Entropy, DFA, Zero Crossing Rate et paramètres de Hjorth.

In [ ]:
if 'raw' in locals() and raw is not None:
    # Define epoch parameters
    tmin = 0      # start of each epoch in seconds
    tmax = 4      # end of each epoch in seconds (4 seconds duration)
    overlap = 0   # No overlap between epochs

    # Generate events for fixed-length epoching
    # The event_id can be arbitrary for fixed-length epochs
    events = mne.make_fixed_length_events(raw, id=1, start=0, duration=tmax, overlap=overlap)

    # Create epochs
    epochs = mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                        preload=True, baseline=None, verbose=False)

    print(f"Number of epochs created: {len(epochs)}")
    print(f"Epochs loaded successfully: {epochs}")

    # Optionally, reject bad epochs based on amplitude thresholds (if not done in preprocessing)
    # epochs.drop_bad(reject=dict(eeg=200e-6)) # e.g., 200 uV threshold
    # print(f"Number of epochs after rejection: {len(epochs)}")
else:
    print("Raw data not loaded. Please ensure the previous cell ran successfully.")

In [ ]:
import antropy as ant
import numpy as np
import pandas as pd

def extract_custom_features(epochs):
    """
    Extracts: Higuchi FD, SVD Entropy, DFA, ZCR, and Hjorth parameters.
    """
    ch_names = epochs.ch_names
    sfreq = epochs.info['sfreq']
    all_epoch_features = []

    print(f"Starting extraction for {len(epochs)} epochs...")

    # Iterate through epochs
    for epoch_idx, data in enumerate(epochs.get_data(copy=False)):
        epoch_feat = {'epoch_idx': epoch_idx}

        # Iterate through channels
        for ch_idx, ch_name in enumerate(ch_names):
            x = data[ch_idx]

            # 1. Higuchi Fractal Dimension
            epoch_feat[f'{ch_name}_hfd'] = ant.higuchi_fd(x)

            # 2. SVD Entropy
            epoch_feat[f'{ch_name}_svd_ent'] = ant.svd_entropy(x, normalize=True)

            # 3. DFA (Detrended Fluctuation Analysis)
            epoch_feat[f'{ch_name}_dfa'] = ant.detrended_fluctuation(x)

            # 4. ZCR (Zero Crossing Rate)
            epoch_feat[f'{ch_name}_zcr'] = ant.num_zerocross(x) / len(x)

            # 5. Hjorth Parameters (Mobility and Complexity)
            # Returns (mobility, complexity)
            hj_mob, hj_comp = ant.hjorth_params(x)
            epoch_feat[f'{ch_name}_hjorth_mobility'] = hj_mob
            epoch_feat[f'{ch_name}_hjorth_complexity'] = hj_comp

        all_epoch_features.append(epoch_feat)
        if (epoch_idx + 1) % 10 == 0:
            print(f"Processed {epoch_idx + 1} epochs...")

    return pd.DataFrame(all_epoch_features)

if 'epochs' in locals():
    final_features_df = extract_custom_features(epochs)
    print("\nFeature extraction complete!")
    display(final_features_df.head())
else:
    print("Please run the segmentation cell first.")

## 2. Extraction de Caractéristiques sur 88 Sujets
Traitement des données du dossier `derivatives`. Pour chaque sujet :
- Chargement des fichiers EEGLAB (.set).
- Segmentation en epochs de 4 secondes.
- Calcul des caractéristiques : Higuchi Fractal Dimension, SVD Entropy, DFA, Zero Crossing Rate et paramètres de Hjorth.

In [ ]:
import os
import mne
import pandas as pd
import antropy as ant
import numpy as np

def extract_features_from_data(epochs, subject_id):
    ch_names = epochs.ch_names
    all_epoch_features = []
    data_array = epochs.get_data(copy=False)

    for epoch_idx, data in enumerate(data_array):
        epoch_feat = {'subject_id': subject_id, 'epoch_idx': epoch_idx}
        for ch_idx, ch_name in enumerate(ch_names):
            x = data[ch_idx]
            epoch_feat[f'{ch_name}_hfd'] = ant.higuchi_fd(x)
            epoch_feat[f'{ch_name}_svd_ent'] = ant.svd_entropy(x, normalize=True)
            epoch_feat[f'{ch_name}_dfa'] = ant.detrended_fluctuation(x)
            epoch_feat[f'{ch_name}_zcr'] = ant.num_zerocross(x) / len(x)
            hj_mob, hj_comp = ant.hjorth_params(x)
            epoch_feat[f'{ch_name}_hjorth_mobility'] = hj_mob
            epoch_feat[f'{ch_name}_hjorth_complexity'] = hj_comp
        all_epoch_features.append(epoch_feat)
    return all_epoch_features

# Chemin des données prétraitées (derivatives) — nécessite data_root
if 'data_root' in globals() and data_root is not None:
    derivatives_path = os.path.join(data_root, 'derivatives')
else:
    raise FileNotFoundError("data_root est introuvable. Assurez-vous que data\\ds004504 existe.")
subjects = sorted([d for d in os.listdir(derivatives_path) if d.startswith('sub-')])

all_results = []
tmin, tmax = 0, 4

print(f"Found {len(subjects)} subjects. Starting processing...")

for sub in subjects:
    try:
        # Path to the preprocessed .set file
        eeg_dir = os.path.join(derivatives_path, sub, 'eeg')
        set_files = [f for f in os.listdir(eeg_dir) if f.endswith('_eeg.set')]

        if not set_files:
            continue

        file_path = os.path.join(eeg_dir, set_files[0])
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

        # Resample to 250Hz if necessary
        if raw.info['sfreq'] > 250:
            raw.resample(250, verbose=False)

        # Segmentation
        events = mne.make_fixed_length_events(raw, id=1, start=0, duration=tmax, overlap=0)
        epochs = mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)

        # Extraction
        sub_features = extract_features_from_data(epochs, sub)
        all_results.extend(sub_features)
        print(f"Processed {sub}: {len(epochs)} epochs")

    except Exception as e:
        print(f"Error processing {sub}: {e}")

# Final DataFrame
full_dataset_features = pd.DataFrame(all_results)
display(full_dataset_features.head())
full_dataset_features.to_csv('alzheimer_eeg_features.csv', index=False)
print(f"Processing finished. Total samples: {len(full_dataset_features)}")

## 4. Analyse de la Matrice et Sauvegarde
Vérification des dimensions de la matrice finale et export au format CSV.

In [ ]:
if 'full_dataset_features' in locals():
    rows, cols = full_dataset_features.shape
    print(f"Dimension de la matrice de caractéristiques : {rows} lignes (epochs) x {cols} colonnes (features)")
    print(f"\nNombre de sujets traités : {full_dataset_features['subject_id'].nunique()}")
    print(f"Nombre de colonnes par électrode : {(cols - 2) // 19} (HFD, SVD_Ent, DFA, ZCR, Mobility, Complexity)")
    display(full_dataset_features.info())
else:
    print("La matrice n'a pas encore été générée. Veuillez terminer l'exécution de la cellule f9767ebf.")

In [ ]:
import os

# Nom du fichier
filename = 'alzheimer_eeg_features.csv'

# Sauvegarde forcée (au cas où)
if 'full_dataset_features' in locals():
    full_dataset_features.to_csv(filename, index=False)
    path = os.path.abspath(filename)
    print(f"Le fichier est sauvegardé avec succès.")
    print(f"Emplacement complet : {path}")
    print(f"Taille du fichier : {os.path.getsize(filename) / 1024 / 1024:.2f} MB")
else:
    print("Erreur : La variable 'full_dataset_features' n'existe pas.")

## 5. Classification Binaire (Alzheimer vs Contrôle)
Évaluation des modèles SVM, KNN et XGBoost par validation croisée stratifiée sur les groupes A et C.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, confusion_matrix

# 1. Charger les labels des participants
if 'data_root' not in globals() or data_root is None:
    raise FileNotFoundError("data_root est introuvable. Assurez-vous que data\\ds004504 existe.")
participants_path = os.path.join(data_root, 'participants.tsv')
participants_df = pd.read_csv(participants_path, sep='\t')
label_map = dict(zip(participants_df['participant_id'], participants_df['Group']))

if 'full_dataset_features' in locals():
    df = full_dataset_features.copy()
    df['target'] = df['subject_id'].map(label_map)

    # FILTRAGE : On garde uniquement A (Alzheimer) et C (Control) pour une classification binaire
    df = df[df['target'].isin(['A', 'C'])]
    print(f"Dataset filtré : {df['target'].value_counts().to_dict()}")

    # Encodage des labels (A -> 1, C -> 0)
    le = LabelEncoder()
    df['target'] = le.fit_transform(df['target'])

    # Features (X) et Target (y)
    X = df.drop(columns=['subject_id', 'epoch_idx', 'target'])
    y = df['target']

    # Gestion des NaNs
    X = X.fillna(X.mean())

    # Normalisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fonctions de score pour classification binaire
    def specificity_score(y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0

    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'sensitivity': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'specificity': make_scorer(specificity_score)
    }

    # 3. Initialisation des modèles
    models = {
        'SVM': SVC(probability=True, kernel='rbf'),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'XGBoost': XGBClassifier(eval_metric='logloss')
    }

    results_list = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("Début de l'évaluation binaire (A vs C)...")
    for name, model in models.items():
        print(f"Calcul pour {name}...")
        cv_results = cross_validate(model, X_scaled, y, cv=skf, scoring=scoring)

        results_list.append({
            'Model': name,
            'Accuracy': cv_results['test_accuracy'].mean(),
            'Precision': cv_results['test_precision'].mean(),
            'Sensitivity': cv_results['test_sensitivity'].mean(),
            'Specificity': cv_results['test_specificity'].mean(),
            'F1 Score': cv_results['test_f1'].mean(),
            'ROC AUC': cv_results['test_roc_auc'].mean()
        })

    perf_df = pd.DataFrame(results_list)
    display(perf_df)
else:
    print("La matrice 'full_dataset_features' est introuvable.")

## 6. Classification Multiclasse (A, C, F)
Évaluation de la capacité des modèles à distinguer les trois groupes (Alzheimer, Contrôle, Fronto-temporal).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, confusion_matrix

# 1. Préparation des labels
if 'data_root' not in globals() or data_root is None:
    raise FileNotFoundError("data_root est introuvable. Assurez-vous que data\\ds004504 existe.")
participants_path = os.path.join(data_root, 'participants.tsv')
participants_df = pd.read_csv(participants_path, sep='\t')
label_map = dict(zip(participants_df['participant_id'], participants_df['Group']))

if 'full_dataset_features' in locals():
    df_3class = full_dataset_features.copy()
    df_3class['target'] = df_3class['subject_id'].map(label_map)

    print(f"Distribution des 3 classes : {df_3class['target'].value_counts().to_dict()}")

    # Encodage (A, C, F)
    le_3 = LabelEncoder()
    y_3 = le_3.fit_transform(df_3class['target'])
    X_3 = df_3class.drop(columns=['subject_id', 'epoch_idx', 'target']).fillna(df_3class.mean(numeric_only=True))

    # Normalisation
    scaler_3 = StandardScaler()
    X_3_scaled = scaler_3.fit_transform(X_3)

    # Métriques Multiclasse (Macro-average)
    scoring_multiclass = {
        'accuracy': 'accuracy',
        'precision_macro': 'precision_macro',
        'recall_macro': 'recall_macro',
        'f1_macro': 'f1_macro',
        'roc_auc_ovr': 'roc_auc_ovr_weighted'
    }

    models_3 = {
        'SVM (3-class)': SVC(probability=True, kernel='rbf'),
        'KNN (3-class)': KNeighborsClassifier(n_neighbors=5),
        'XGBoost (3-class)': XGBClassifier(eval_metric='mlogloss', num_class=3)
    }

    results_3class = []
    skf_3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("Début de l'évaluation multiclasse (A, C, F)...")
    for name, model in models_3.items():
        print(f"Calcul pour {name}...")
        cv_res = cross_validate(model, X_3_scaled, y_3, cv=skf_3, scoring=scoring_multiclass)

        results_3class.append({
            'Model': name,
            'Accuracy': cv_res['test_accuracy'].mean(),
            'Precision (Macro)': cv_res['test_precision_macro'].mean(),
            'Recall/Sens. (Macro)': cv_res['test_recall_macro'].mean(),
            'F1 Score (Macro)': cv_res['test_f1_macro'].mean(),
            'ROC AUC (OvR)': cv_res['test_roc_auc_ovr'].mean()
        })

    perf_3class_df = pd.DataFrame(results_3class)
    print("\nRésultats de la classification à 3 classes :")
    display(perf_3class_df)
else:
    print("La matrice 'full_dataset_features' est manquante.")

## 7. Visualisation des performances et export des figures — sauvegarde sous results/figures (racine du projet)
Cette section génère des figures explicatives des performances des modèles et les enregistre sous `results/figures` à la racine du projet:
- Classification binaire (A vs C): barplot des métriques, matrice de confusion du meilleur modèle, courbes ROC.
- Classification multiclasse (A, C, F): barplot des métriques macro et matrice de confusion du meilleur modèle.

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Dossier de sortie et style (7.0 — Préparation)
# Enregistre à la racine du dépôt: ..\results\figures depuis le notebook
fig_dir = os.path.join('..', 'results', 'figures')
os.makedirs(fig_dir, exist_ok=True)
print(f"Dossier des figures: {os.path.abspath(fig_dir)}")

sns.set(style='whitegrid')


Dossier des figures: C:\Users\USER\PycharmProjects\EEG_ML_pipelines\results\figures


### 7.1 — Binaire: Barplot des métriques — résumé visuel des scores moyens

In [25]:
if 'perf_df' in globals() and isinstance(perf_df, pd.DataFrame) and not perf_df.empty:

    # Liste des métriques souhaitées avec gestion des variantes de nommage
    metrics_cols = [
        'Accuracy',
        'Precision',
        'Sensitivity',      # Standard
        'Sensibilité',      # Variante française
        'Recall',           # Synonyme de Sensitivity
        'Specificity',
        'Spécificité',      # Variante française
        'F1 Score',
        'F1',               # Variante courte
        'ROC AUC',
        'AUC'               # Variante courte
    ]

    # Détection des colonnes disponibles (insensible à la casse)
    available_metrics = []
    perf_df_cols_lower = [c.lower().replace(' ', '').replace('_', '') for c in perf_df.columns]

    for m in metrics_cols:
        # Recherche insensible à la casse et aux espaces/underscores
        m_normalized = m.lower().replace(' ', '').replace('_', '')
        if m_normalized in perf_df_cols_lower:
            # Récupérer le nom original exact dans le DataFrame
            idx = perf_df_cols_lower.index(m_normalized)
            original_name = perf_df.columns[idx]
            available_metrics.append(original_name)

    # Suppression des doublons tout en préservant l'ordre
    seen = set()
    available_metrics = [x for x in available_metrics if not (x in seen or seen.add(x))]

    if available_metrics:
        plt.figure(figsize=(12, 6))  # Légèrement plus large pour accommoder Sensitivity

        # Vérifier que 'Model' existe, sinon utiliser l'index
        if 'Model' in perf_df.columns:
            perf_df_plot = perf_df.set_index('Model')[available_metrics]
        else:
            perf_df_plot = perf_df[available_metrics]
            print("Note: colonne 'Model' non trouvée, utilisation de l'index par défaut.")

        # Création du barplot
        ax = perf_df_plot.plot(
            kind='bar',
            ylim=(0, 1.05),  # Marge légère au-dessus de 1
            rot=45,          # Rotation des labels pour lisibilité
            figsize=(12, 6),
            colormap='viridis',
            edgecolor='black',
            linewidth=0.5
        )

        plt.title('Classification binaire — Métriques moyennes par modèle', fontsize=14, pad=20)
        plt.ylabel('Score', fontsize=12)
        plt.xlabel('Modèle', fontsize=12)
        plt.legend(
            title='Métrique',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            frameon=True,
            shadow=True
        )

        # Ajout de grille pour faciliter la lecture
        plt.grid(axis='y', alpha=0.3, linestyle='--')

        # Annotation des valeurs sur les barres (optionnel mais utile)
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', rotation=90, padding=3, fontsize=8)

        plt.tight_layout()

        # Sauvegarde
        out_path = os.path.join(fig_dir, 'binary_metrics_bar.png')
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()

        print(f"Figure sauvegardée: {out_path}")
        print(f"Métriques affichées: {', '.join(available_metrics)}")

        # Affichage spécifique pour vérifier que Sensitivity est bien là
        sensitivity_found = [m for m in available_metrics if 'sens' in m.lower()]
        if sensitivity_found:
            print(f"✓ Sensitivity détectée: {sensitivity_found}")
        else:
            print("⚠ Attention: Sensitivity non trouvée dans les métriques disponibles")

    else:
        print("Aucune métrique disponible pour le barplot binaire.")
        print(f"Colonnes présentes dans perf_df: {list(perf_df.columns)}")

else:
    print("Section binaire: perf_df indisponible, barplot non généré.")

Figure sauvegardée: ..\results\figures\binary_metrics_bar.png
Métriques affichées: Accuracy, Precision, Sensitivity, Specificity, F1 Score, ROC AUC
✓ Sensitivity détectée: ['Sensitivity']


<Figure size 1200x600 with 0 Axes>

### 7.2 — Binaire: Matrice de confusion du meilleur modèle — répartition des prédictions en CV

In [19]:
if all(name in globals() for name in ['perf_df', 'X_scaled', 'y', 'models', 'skf']) \
   and isinstance(perf_df, pd.DataFrame) and not perf_df.empty \
   and isinstance(models, dict) and len(models) > 0 \
   and isinstance(skf, StratifiedKFold):

    rank_metric = 'ROC AUC' if 'ROC AUC' in perf_df.columns else 'Accuracy'
    best_row = perf_df.sort_values(by=rank_metric, ascending=False).iloc[0]
    best_model_name = best_row['Model']
    best_model = models[best_model_name]
    print(f"Meilleur modèle (binaire) selon {rank_metric}: {best_model_name}")

    y_pred_cv = cross_val_predict(best_model, X_scaled, y, cv=skf)
    cm = confusion_matrix(y, y_pred_cv, labels=[0, 1])
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(4.8, 4))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', cbar=False,
                xticklabels=['C', 'A'], yticklabels=['C', 'A'])
    plt.title(f'Matrice de confusion (CV) — {best_model_name}')
    plt.xlabel('Prédit')
    plt.ylabel('Vrai')
    plt.tight_layout()
    out_path = os.path.join(fig_dir, 'binary_confusion_best.png')
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Figure sauvegardée: {out_path}")
else:
    print("Section binaire: données/modèles indisponibles pour la matrice de confusion.")


Meilleur modèle (binaire) selon ROC AUC: XGBoost
Figure sauvegardée: ..\results\figures\binary_confusion_best.png


### 7.3 — Binaire: Courbes ROC (validation croisée) — performances discriminantes par modèle

In [23]:
# 7.3a — Pré-calcul des probabilités CV et courbes ROC (lourd)
if all(name in globals() for name in ['X_scaled', 'y', 'models', 'skf']) \
   and isinstance(models, dict) and len(models) > 0 \
   and isinstance(skf, StratifiedKFold):
    roc_curves_bin = {}
    for name, clf in models.items():
        try:
            y_proba = cross_val_predict(clf, X_scaled, y, cv=skf, method='predict_proba')[:, 1]
            fpr, tpr, _ = roc_curve(y, y_proba)
            roc_auc = auc(fpr, tpr)
            roc_curves_bin[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
            print(f"ROC calculée pour {name}: AUC={roc_auc:.3f}")
        except Exception as e:
            print(f"ROC non disponible pour {name}: {e}")
    if len(roc_curves_bin) == 0:
        print("Aucune courbe ROC n'a pu être calculée.")
else:
    print("Section binaire: données/modèles indisponibles pour le calcul des courbes ROC.")


ROC calculée pour SVM: AUC=0.956
ROC calculée pour KNN: AUC=0.938
ROC calculée pour XGBoost: AUC=0.972


In [24]:
# 7.3b — Tracé et sauvegarde des courbes ROC (léger)
if 'roc_curves_bin' in globals() and isinstance(roc_curves_bin, dict) and len(roc_curves_bin) > 0:
    plt.figure(figsize=(6, 5))
    for name, d in roc_curves_bin.items():
        plt.plot(d['fpr'], d['tpr'], label=f"{name} (AUC={d['auc']:.2f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taux de faux positifs (FPR)')
    plt.ylabel('Taux de vrais positifs (TPR)')
    plt.title('Classification binaire — Courbes ROC (CV)')
    plt.legend(loc='lower right')
    plt.tight_layout()
    out_path = os.path.join(fig_dir, 'binary_roc_curves.png')
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Figure sauvegardée: {out_path}")
else:
    print("Section binaire: dictionnaire 'roc_curves_bin' indisponible pour le tracé des courbes ROC.")


Figure sauvegardée: ..\results\figures\binary_roc_curves.png


### 7.4 — Multiclasse: Barplot des métriques (macro) — comparaison moyenne des modèles

In [21]:
if 'perf_3class_df' in globals() and isinstance(perf_3class_df, pd.DataFrame) and not perf_3class_df.empty:
    metrics_3_cols = ['Accuracy', 'Precision (Macro)', 'Recall/Sens. (Macro)', 'F1 Score (Macro)', 'ROC AUC (OvR)']
    available_3 = [m for m in metrics_3_cols if m in perf_3class_df.columns]
    if available_3:
        plt.figure(figsize=(10, 5))
        perf3_plot = perf_3class_df.set_index('Model')[available_3]
        perf3_plot.plot(kind='bar', ylim=(0, 1), rot=15, figsize=(10, 5), colormap='magma')
        plt.title('Multiclasse — Métriques (macro) par modèle')
        plt.ylabel('Score')
        plt.xlabel('Modèle')
        plt.legend(title='Métrique', bbox_to_anchor=(1.02, 1), loc='upper left')
        plt.tight_layout()
        out_path = os.path.join(fig_dir, 'multiclass_metrics_bar.png')
        plt.savefig(out_path, dpi=150)
        plt.close()
        print(f"Figure sauvegardée: {out_path}")
    else:
        print("Aucune métrique disponible pour le barplot multiclasse.")
else:
    print("Section multiclasse: perf_3class_df indisponible, barplot non généré.")


Figure sauvegardée: ..\results\figures\multiclass_metrics_bar.png


<Figure size 1000x500 with 0 Axes>

### 7.5 — Multiclasse: Matrice de confusion du meilleur modèle — erreurs par classe pour le top modèle

In [22]:
if all(name in globals() for name in ['perf_3class_df', 'X_3_scaled', 'y_3', 'models_3', 'skf_3']) \
   and isinstance(perf_3class_df, pd.DataFrame) and not perf_3class_df.empty \
   and isinstance(models_3, dict) and len(models_3) > 0 \
   and isinstance(skf_3, StratifiedKFold):

    available_3 = ['Accuracy', 'Precision (Macro)', 'Recall/Sens. (Macro)', 'F1 Score (Macro)', 'ROC AUC (OvR)']
    rank3 = 'Accuracy' if 'Accuracy' in perf_3class_df.columns else available_3[0]
    best3_row = perf_3class_df.sort_values(by=rank3, ascending=False).iloc[0]
    best3_name = best3_row['Model']
    best3_model = models_3[best3_name]
    print(f"Meilleur modèle (multiclasse) selon {rank3}: {best3_name}")

    y3_pred = cross_val_predict(best3_model, X_3_scaled, y_3, cv=skf_3)
    labels_sorted = sorted(np.unique(y_3))
    try:
        class_names = list(le_3.classes_)
        mapping = {i: name for i, name in enumerate(class_names)}
        tick_labels = [mapping[i] if i in mapping else str(i) for i in labels_sorted]
    except Exception:
        tick_labels = [str(i) for i in labels_sorted]

    cm3 = confusion_matrix(y_3, y3_pred, labels=labels_sorted)
    cm3_norm = cm3 / cm3.sum(axis=1, keepdims=True)

    plt.figure(figsize=(5.2, 4.5))
    sns.heatmap(cm3_norm, annot=True, fmt='.2f', cmap='Purples', cbar=False,
                xticklabels=tick_labels, yticklabels=tick_labels)
    plt.title(f'Matrice de confusion (CV) — {best3_name}')
    plt.xlabel('Prédit')
    plt.ylabel('Vrai')
    plt.tight_layout()
    out_path = os.path.join(fig_dir, 'multiclass_confusion_best.png')
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"Figure sauvegardée: {out_path}")
else:
    print("Section multiclasse: données/modèles indisponibles pour la matrice de confusion.")


Meilleur modèle (multiclasse) selon Accuracy: XGBoost (3-class)
Figure sauvegardée: ..\results\figures\multiclass_confusion_best.png
